# sdm-client quickstart

One client, two models, and the model is an argument rather than a method name.

Everything below runs against a NIM you point it at. Set the two URLs to match
your deployment; the tabular half needs only the base install, the relational
half needs `pip install sdm-client[relational]`.


In [ ]:
import pandas as pd
from sdm_client import (
    KumoRelational,
    KumoTabular,
    PredictiveQuery,
    Request,
    Schema,
    StructuredDataClient,
)

# Point these at your own deployment.
TABULAR_URL = 'http://localhost:8000'
RELATIONAL_URL = 'http://localhost:8001'

## One table

You have rows where the answer is known, and rows you want scored. The rows
being scored need not appear in the first half — that is the point of the
model.


In [ ]:
labelled = pd.DataFrame(
    {
        'age': [31.0, 44.5, 28.0, 51.0, 39.0, 47.0, 25.0, 58.0],
        'spend': [120.0, 430.0, 90.0, 510.0, 260.0, 480.0, 70.0, 620.0],
        'churn': [0, 1, 0, 1, 0, 1, 0, 1],
    }
)
to_score = pd.DataFrame({'age': [33.0, 47.0], 'spend': [150.0, 480.0]})

request = Request(
    context=labelled,
    predict=to_score,
    target='churn',
    task='classification',
)
request

In [ ]:
with StructuredDataClient(url=TABULAR_URL) as client:
    print('the endpoint serves:', client.models())
    result = client.predict(request, model=KumoTabular())

result.to_pandas()

Join on the key the result carries, or on `row_index`. **Never on row
order** — the endpoint is not required to answer in the order you asked, and a
request split across several calls certainly will not.


In [ ]:
answers = result.to_pandas()
to_score.reset_index(names='row_index').merge(answers, on='row_index')

## Several related tables

Describe the tables, ask in PQL, and name the entities you want scored.


In [ ]:
n_users = 40
users = pd.DataFrame({'user_id': range(n_users)})

rows, order_id = [], 0
for user in range(n_users):
    busy = user % 2 == 0
    for k in range(40 if busy else 4):
        rows.append(
            {
                'order_id': order_id,
                'user_id': user,
                'price': 10.0 + (user % 7) * 3 + k,
                # A 30-day aggregation needs 30 days of history before the anchor.
                'ts': pd.Timestamp('2024-01-01')
                + pd.Timedelta(days=(k * 5 if busy else k * 50) + user % 9),
            }
        )
        order_id += 1
orders = pd.DataFrame(rows)

schema = Schema.from_data({'users': users, 'orders': orders})
schema

Everything is inferred — dtypes, semantic types, keys, time columns and the
relationship between the two tables. Override only where inference is wrong.

A code stored as a number reads as a number, so a postcode or a status code
needs `schema["t"]["col"].stype = "categorical"`.


In [ ]:
schema['orders'], schema['users'].primary_key, schema.relationships

In [ ]:
query = PredictiveQuery(
    'PREDICT COUNT(orders.*, 0, 30, days) FOR EACH users.user_id',
    schema,
)

# The gathering step is separate because it is the expensive one: it resolves
# the query, finds labelled examples in history, samples each entity's
# neighbourhood and encodes the result. Nothing is sent here.
relational_request = query.materialize(indices=[0, 1, 2, 3])
relational_request

In [ ]:
with StructuredDataClient(url=RELATIONAL_URL, timeout=600.0) as client:
    relational_result = client.predict(
        relational_request, model=KumoRelational()
    )

relational_result.to_pandas()

Even-numbered users order roughly ten times as often as odd ones, and the
predictions should say so.

## Asking one context several questions

Uploading a context once and scoring against it repeatedly is what `fit` is
for. The pinned half is model, task, schema and context; later calls send only
the rows.


In [ ]:
with StructuredDataClient(url=RELATIONAL_URL, timeout=600.0) as client:
    with client.fit(relational_request, model=KumoRelational()) as fitted:
        print('session:', fitted.session_id)
        display(fitted.predict(relational_request).to_pandas())

## What a model will refuse

A model says no *before* anything is sent, because what it may be asked is
declared as data rather than discovered from a 422.


In [ ]:
from sdm_client import UnsupportedRequestError

with StructuredDataClient(url=TABULAR_URL) as client:
    try:
        client.predict(relational_request, model=KumoTabular())
    except UnsupportedRequestError as error:
        print(error)